# Diagnostik residual HBP selektif SNI (murah)
Warm-start dari SNIB1 seed 42. Encoder, fusi multiresolusi, dan classifier dasar dibekukan. Hanya dua residual head capacity-matched yang dilatih selama 10 epoch: projected-GAP control (SNIDG) dan HBP (SNIDH). Validation only; test tetap terkunci.

In [ ]:
REPO_REF = 'agent/sni-instance-crops'
DRIVE_DATA_FOLDER = 'coffee-sni-instance-crop-v1'
BASELINE_FOLDER = 'sni-mrenet-v1'
DIAGNOSTIC_FOLDER = 'sni-selective-hbp-diagnostic-v1'

In [ ]:
# SETUP REPO DAN DRIVE
import csv, json, shutil, subprocess, sys, tarfile, time
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')
repo = Path('/content/coffee-bean-classification')
url = 'https://github.com/ediprin/coffee-bean-classification.git'
def run(command, cwd=None):
    print('\n$', ' '.join(map(str, command)), flush=True)
    return subprocess.run(list(map(str, command)), cwd=cwd, check=True)
if (repo / '.git').is_dir():
    run(['git', 'fetch', 'origin', REPO_REF], cwd=repo)
    run(['git', 'checkout', REPO_REF], cwd=repo)
    run(['git', 'pull', '--ff-only', 'origin', REPO_REF], cwd=repo)
else:
    if repo.exists(): shutil.rmtree(repo)
    run(['git', 'clone', '--branch', REPO_REF, url, repo])
run([sys.executable, '-m', 'pip', 'install', '-q', '-e', repo])

In [ ]:
# PULIHKAN DATASET DARI SHARD DRIVE
data_root = Path('/content/sni-instance-crops')
drive_data = Path('/content/drive/MyDrive') / DRIVE_DATA_FOLDER
if not (data_root / 'audit.json').is_file():
    shards = sorted((drive_data / 'shards').glob('crop_shard_*.tar'))
    assert (drive_data / 'complete.json').is_file() and shards, 'Backup dataset belum lengkap.'
    data_root.mkdir(parents=True, exist_ok=True)
    for i, shard in enumerate(shards, 1):
        with tarfile.open(shard, 'r') as archive: archive.extractall(data_root, filter='data')
        if i % 5 == 0 or i == len(shards): print(f'RESTORE {i}/{len(shards)}')
    for name in ('audit.json', 'manifest.csv'): shutil.copy2(drive_data / name, data_root / name)
audit = json.loads((data_root / 'audit.json').read_text())
assert audit['status'] == 'complete' and audit['num_classes'] == 21
assert audit['generated_cross_split_identity_count'] == 0 and audit['test_locked'] is True
baseline_root = Path('/content/drive/MyDrive') / BASELINE_FOLDER
output_root = Path('/content/drive/MyDrive') / DIAGNOSTIC_FOLDER
baseline = baseline_root / 'outputs/SNIB1_seed42/best.pt'
baseline_metrics = baseline_root / 'val_reports/SNIB1_seed42/metrics.json'
assert baseline.is_file() and baseline_metrics.is_file(), 'Checkpoint/report SNIB1 seed42 tidak ada.'
output_root.mkdir(parents=True, exist_ok=True)
print('DATA:', data_root, '| BASELINE:', baseline, '| OUTPUT:', output_root)

In [ ]:
# JALANKAN DIAGNOSTIK 2 x 10 EPOCH DENGAN HEARTBEAT
command = [sys.executable, '-u', '-m', 'bilinear_lmmd.experiments.run_sni_selective_residual_diagnostic', '--data-root', str(data_root), '--baseline-root', str(baseline_root), '--output-root', str(output_root), '--evaluation-split', 'val']
print('MENJALANKAN:', ' '.join(command), flush=True)
process = subprocess.Popen(command, cwd=repo)
started = time.monotonic()
while process.poll() is None:
    status = []
    for code in ('SNIDG', 'SNIDH'):
        path = output_root / f'outputs/{code}_seed42/history.json'
        if path.is_file():
            try: status.append(f'{code}={len(json.loads(path.read_text()))}/10')
            except Exception: status.append(f'{code}=saving')
    print(f'[DIAGNOSTIC {(time.monotonic()-started)/60:.1f} menit]', ', '.join(status) or 'inisialisasi', flush=True)
    time.sleep(60)
assert process.wait() == 0, 'Diagnostik gagal; baca traceback di atas.'

In [ ]:
# TAMPILKAN PUTUSAN
path = output_root / 'val_reports/selective_residual_diagnostic.json'
report = json.loads(path.read_text())
for name, row in report['decisions'].items(): print(name, row['decision'], row['criteria'])
print('FINAL:', report['final_decision'])
print('Test dibuka:', report['test_opened'])
print('Jika STOP, residual HBP dihentikan. Jangan jalankan seed/test lain.')